# Disk hit acceptance in generated eta and phi

This notebook maps the generated primary particles that have at least one directly associated hit on each disk. It bins generated $\eta$ and $\phi$; it does not project a particle to a disk plane. The denominator is all generated primary $\pi^+$ particles in the selected momentum interval and the numerator is the subset with a hit on that disk. Each primary counts once per disk.

The first available sample is vertex $z=0$ mm. Change `VERTICES_MM` after extracting the other vertex datasets.

## Configure and run

The default momentum intervals span the generated 0.5–20 GeV range. The first interval, 0.5–1.5 GeV, gives a useful initial view around 1 GeV. Plots show both hit counts and hit fraction. Fraction bins with fewer than `MINIMUM_BIN_DENOMINATOR` generated particles are masked.

Each scan writes a fresh output directory. Change `OUTPUT_TAG` before rerunning with different settings.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / 'plot_hit_eta_phi_acceptance.py').is_file():
    SCRIPT_DIR = Path('proj_svt_disk_tiling/simrec/acc_study').resolve()
if not (SCRIPT_DIR / 'plot_hit_eta_phi_acceptance.py').is_file():
    raise FileNotFoundError('Run from this notebook directory or the EPIC repository root.')

EXTRACTED_ROOT = SCRIPT_DIR / 'extracted' / 'acceptance_v1'
OUTPUT_TAG = 'hit_eta_phi_v2'
OUTPUT_DIR = SCRIPT_DIR / 'results' / OUTPUT_TAG

VERTICES_MM = [0]                 # Available now: [0]; add -10 and +10 after extraction.
MOMENTUM_BINS_GEV = [
    (0.5, 1.5),
    (1.5, 3.0),
    (3.0, 5.0),
    (5.0, 10.0),
    (10.0, 20.01),                # Include the generated endpoint at 20 GeV.
]
ETA_BIN_WIDTH = 0.1
MINIMUM_BIN_DENOMINATOR = 20
RUN_SCAN = False                # Set True to make all requested maps.

In [ ]:
command = [
    sys.executable, str(SCRIPT_DIR / 'plot_hit_eta_phi_acceptance.py'),
    '--extracted-root', str(EXTRACTED_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--vertices', *map(str, VERTICES_MM),
    '--eta-bin-width', str(ETA_BIN_WIDTH),
    '--minimum-bin-denominator', str(MINIMUM_BIN_DENOMINATOR),
]
for low, high in MOMENTUM_BINS_GEV:
    command.extend(['--momentum-bin', str(low), str(high)])
print(' '.join(command))
if RUN_SCAN:
    subprocess.run(command, check=True)
else:
    print('Scan not run. Set RUN_SCAN = True when ready.')

## Review the first momentum interval

Each PNG is a 2×5 grid: ED0–ED4 on the left and HD0–HD4 on the right. The count map shows generated particles associated with a disk hit; the fraction map divides those counts by all generated particles in the same $(\eta,\phi)$ bin. Use the count map to judge statistics and the fraction map to locate the populated acceptance region.

In [ ]:
first_low, first_high = MOMENTUM_BINS_GEV[0]
tag = f'p{first_low:g}_to_{first_high:g}'.replace('.', 'p')
vertex_tag = '000' if VERTICES_MM[0] == 0 else f"{'p' if VERTICES_MM[0] > 0 else 'm'}{abs(VERTICES_MM[0]):03d}"
stem = f'hit_eta_phi_vz_{vertex_tag}mm_{tag}'
count_plot = OUTPUT_DIR / f'{stem}_counts.png'
fraction_plot = OUTPUT_DIR / f'{stem}_fraction.png'
if count_plot.is_file() and fraction_plot.is_file():
    display(Image(filename=str(count_plot)))
    display(Image(filename=str(fraction_plot)))
    summary = pd.read_csv(OUTPUT_DIR / 'hit_eta_phi_summary.csv')
    selected_rows = (summary.vertex_z_mm == VERTICES_MM[0]) & (summary.momentum_min_GeV == first_low)
    display(summary.loc[selected_rows, ['disk', 'generated_particles', 'particles_with_disk_hit',
                                       'integrated_hit_fraction', 'eta_phi_bins_unmasked']])
else:
    print(f'No plots yet. Run the scan to create {OUTPUT_DIR}.')